# 🪙 Crypto Market Microstructure — Complete EDA
## 2022–2024 Bear → Crab → Bull Cycle Analysis

**Dataset:** Crypto Market Microstructure (2022–2024)  
**Author:** Sergey Nefedov | [github.com/Sergpreneur](https://github.com/Sergpreneur)

---

### What this notebook covers
1. 📈 Price & regime analysis across the full 2022–2024 cycle
2. 💸 Funding rates as a sentiment & mean-reversion signal
3. 💥 Liquidation cascade anatomy — LUNA and FTX collapses
4. 🔗 On-chain metrics: MVRV, NVT, SOPR cycle analysis
5. 🔀 Cross-exchange arbitrage spreads during stress events
6. 🤖 Regime classification — price features vs on-chain features

> **Key thesis:** Crypto microstructure data contains signals unavailable in traditional finance —  
> funding rates, on-chain flows, and liquidation cascades — that explain price dynamics better than price alone.


## 0. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 130,
    'axes.facecolor': '#0d1117', 'figure.facecolor': '#0d1117',
    'axes.edgecolor': '#30363d', 'axes.labelcolor': '#c9d1d9',
    'xtick.color': '#8b949e',    'ytick.color': '#8b949e',
    'text.color': '#c9d1d9',     'grid.color': '#21262d',
    'grid.alpha': 0.6,           'axes.spines.top': False,
    'axes.spines.right': False,
})

COLORS = {
    'bear':'#f85149','crab':'#388bfd','bull':'#3fb950',
    'btc':'#f7931a','eth':'#627eea','sol':'#9945ff',
    'bnb':'#f3ba2f','matic':'#8247e5','neutral':'#8b949e',
}

import os

# Auto-detect dataset path
def find_path():
    base = '/kaggle/input/'
    for d in os.listdir(base):
        if 'crypto' in d.lower() or 'microstructure' in d.lower():
            return base + d + '/'
    # fallback: list available datasets
    print('Available datasets:', os.listdir(base))
    return base + os.listdir(base)[0] + '/'

PATH = find_path()
print(f'Using dataset path: {PATH}')
print('Files found:', os.listdir(PATH))

ohlcv    = pd.read_csv(PATH + 'ohlcv_daily.csv',            parse_dates=['date'])
funding  = pd.read_csv(PATH + 'funding_rates.csv',          parse_dates=['datetime'])
liqs     = pd.read_csv(PATH + 'liquidations.csv',           parse_dates=['date'])
onchain  = pd.read_csv(PATH + 'on_chain_metrics.csv',       parse_dates=['date'])
cex      = pd.read_csv(PATH + 'cross_exchange_spread.csv',  parse_dates=['date'])
regime_f = pd.read_csv(PATH + 'regime_features.csv',        parse_dates=['date'])

print(f"✅ OHLCV:            {ohlcv.shape[0]:>7,} rows | {ohlcv['symbol'].nunique()} assets | {ohlcv['date'].min().date()} → {ohlcv['date'].max().date()}")
print(f"✅ Funding rates:    {funding.shape[0]:>7,} rows | 8h intervals, 4 assets")
print(f"✅ Liquidations:     {liqs.shape[0]:>7,} rows | ${liqs['liq_usd'].sum()/1e9:.1f}B total liquidated")
print(f"✅ On-chain metrics: {onchain.shape[0]:>7,} rows | {list(onchain['symbol'].unique())}")
print(f"✅ Cross-exchange:   {cex.shape[0]:>7,} rows | {list(cex['exchange'].unique())}")
print(f"✅ Regime features:  {regime_f.shape[0]:>7,} rows | {regime_f['regime'].value_counts().to_dict()}")


---
## 1. 📈 Price & Regime Analysis

Three distinct periods in the 2022–2024 dataset:
- **Bear (2022):** LUNA collapse (−35% BTC, May) → 3AC → FTX collapse (−28% BTC, Nov)
- **Crab (2023):** Slow recovery, SEC lawsuits against Binance/Coinbase, USDC depeg
- **Bull (2024):** Bitcoin ETF approval (Jan), halving (Apr), breakout to new ATH


In [ ]:
btc = ohlcv[ohlcv['symbol']=='BTC'].copy().sort_values('date')

SYMBOLS    = ['BTC','ETH','SOL','BNB','MATIC']
SYM_COLORS = {'BTC':'#f7931a','ETH':'#627eea','SOL':'#9945ff','BNB':'#f3ba2f','MATIC':'#8247e5'}
REG_COLORS = {'bear':'#f85149','crab':'#388bfd','bull':'#3fb950'}

fig, axes = plt.subplots(3,1, figsize=(16,14), gridspec_kw={'height_ratios':[3,1,1]})

# Panel 1: Indexed prices
ax = axes[0]
for sym in SYMBOLS:
    sub  = ohlcv[ohlcv['symbol']==sym].sort_values('date')
    norm = sub['close'] / sub['close'].iloc[0] * 100
    ax.plot(sub['date'], norm, label=sym, color=SYM_COLORS[sym], linewidth=1.8, alpha=0.9)

prev_r = btc['regime'].iloc[0]; start_d = btc['date'].iloc[0]
for _, row in btc.iterrows():
    if row['regime'] != prev_r:
        ax.axvspan(start_d, row['date'], alpha=0.08, color=REG_COLORS[prev_r])
        start_d = row['date']; prev_r = row['regime']
ax.axvspan(start_d, btc['date'].iloc[-1], alpha=0.08, color=REG_COLORS[prev_r])

shocks = btc[btc['shock_event'].notna()][['date','shock_event','close']].drop_duplicates('shock_event')
for _, s in shocks.iterrows():
    nv = s['close'] / btc['close'].iloc[0] * 100
    ax.axvline(s['date'], color='#f85149', linewidth=1, linestyle='--', alpha=0.6)
    ax.text(s['date'], nv*1.06, s['shock_event'].replace('_',' '), fontsize=7, color='#f85149', ha='center', rotation=15)

patches = [mpatches.Patch(color=c, alpha=0.4, label=r.capitalize()) for r,c in REG_COLORS.items()]
lines   = [plt.Line2D([0],[0], color=SYM_COLORS[s], linewidth=2, label=s) for s in SYMBOLS]
ax.legend(handles=patches+lines, loc='upper right', fontsize=8, ncol=2)
ax.axhline(100, color='#8b949e', linewidth=0.8, linestyle='--', alpha=0.5)
ax.set_title('Crypto Asset Performance — Indexed to 100 (Jan 2022)', fontsize=13, pad=12)
ax.set_ylabel('Indexed Price'); ax.grid(True, alpha=0.3)

# Panel 2: BTC daily returns
ax2 = axes[1]
pos = btc['daily_return'] >= 0
ax2.bar(btc['date'][pos],  btc['daily_return'][pos]*100,  color='#3fb950', alpha=0.7, width=1)
ax2.bar(btc['date'][~pos], btc['daily_return'][~pos]*100, color='#f85149', alpha=0.7, width=1)
ax2.axhline(0, color='#8b949e', linewidth=0.8)
ax2.set_ylabel('Return (%)'); ax2.set_title('BTC Daily Returns', fontsize=10); ax2.grid(True, alpha=0.3)

# Panel 3: Volume
ax3 = axes[2]
vol = btc['volume_usd']/1e9
ax3.fill_between(btc['date'], vol, alpha=0.6, color='#f7931a')
ax3.plot(btc['date'], vol, color='#f7931a', linewidth=0.8)
ax3.set_ylabel('Volume ($B)'); ax3.set_title('BTC Trading Volume', fontsize=10); ax3.grid(True, alpha=0.3)

plt.tight_layout(pad=2)
plt.savefig('price_analysis.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()

print("\n=== BTC Regime Performance ===")
for r in ['bear','crab','bull']:
    sub = btc[btc['regime']==r]
    tr  = (1+sub['daily_return']).prod()-1
    vol = sub['daily_return'].std()*np.sqrt(365)
    print(f"{r.upper():6s}: {len(sub):3d} days | return {tr:+.1%} | ann.vol {vol:.1%} | Sharpe {tr/vol:.2f}")


---
## 2. 💸 Funding Rates — Crypto's Unique Sentiment Signal

Perpetual futures funding settled every 8 hours:
- **Positive funding** → longs pay shorts → crowded long → **mean-reversion signal**
- **Negative funding** → shorts pay longs → crowded short → **potential squeeze**


In [ ]:
btc_fr = funding[funding['symbol']=='BTC'].copy().sort_values('datetime')

fig, axes = plt.subplots(2,2, figsize=(16,10))

# Panel 1: Time series
ax = axes[0,0]
fr_d = btc_fr.set_index('datetime')['funding_rate'].resample('D').mean()
pm = fr_d >= 0
ax.fill_between(fr_d.index, fr_d*100, 0, where=pm,  alpha=0.6, color='#3fb950', label='Positive (longs pay)')
ax.fill_between(fr_d.index, fr_d*100, 0, where=~pm, alpha=0.6, color='#f85149', label='Negative (shorts pay)')
ax.axhline(0.03,  color='#f85149', linewidth=0.8, linestyle='--', alpha=0.7, label='Extreme + threshold')
ax.axhline(-0.03, color='#3fb950', linewidth=0.8, linestyle='--', alpha=0.7)
ax.set_title('BTC Perpetual Funding Rate — Daily Average (%)', fontsize=11)
ax.set_ylabel('Funding Rate (%)'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# Panel 2: Box by regime
ax = axes[0,1]
btc_fr['date'] = btc_fr['datetime'].dt.date.astype(str)
btc_reg = btc[['date','regime']].copy(); btc_reg['date'] = btc_reg['date'].astype(str)
frm = btc_fr.merge(btc_reg, on='date', how='left')
data_r = [frm[frm['regime']==r]['funding_rate']*100 for r in ['bear','crab','bull']]
bp = ax.boxplot(data_r, labels=['Bear','Crab','Bull'], patch_artist=True,
                medianprops=dict(color='white', linewidth=2))
for patch, color in zip(bp['boxes'], ['#f85149','#388bfd','#3fb950']):
    patch.set_facecolor(color); patch.set_alpha(0.7)
ax.axhline(0, color='#8b949e', linewidth=0.8, linestyle='--')
ax.set_title('Funding Distribution by Regime', fontsize=11)
ax.set_ylabel('8h Funding Rate (%)'); ax.grid(True, alpha=0.3, axis='y')

# Panel 3: IC analysis
ax = axes[1,0]
fr_df = fr_d.reset_index(); fr_df.columns = ['date','avg_fr']
fr_df['date'] = fr_df['date'].dt.normalize()
bm = btc[['date','daily_return']].copy(); bm['fwd'] = bm['daily_return'].shift(-1)
bm = bm.merge(fr_df, on='date', how='inner').dropna()
bm['decile'] = pd.qcut(bm['avg_fr'], 10, labels=False, duplicates='drop')
dr = bm.groupby('decile')['fwd'].mean()*100
ax.bar(dr.index, dr.values, color=['#f85149' if v<0 else '#3fb950' for v in dr.values], alpha=0.8)
ax.axhline(0, color='#8b949e', linewidth=0.8)
ax.set_title('Mean Next-Day Return by Funding Decile', fontsize=11)
ax.set_xlabel('Decile (D1=most negative → D10=most positive)')
ax.set_ylabel('Mean Forward Return (%)')
ic, pval = stats.spearmanr(bm['avg_fr'], bm['fwd'])
ax.text(0.05, 0.92, f'Spearman IC = {ic:.3f}  (p={pval:.3f})',
        transform=ax.transAxes, fontsize=9,
        bbox=dict(boxstyle='round', facecolor='#21262d', alpha=0.8))
ax.grid(True, alpha=0.3, axis='y')

# Panel 4: Annualised by asset
ax = axes[1,1]
sf = funding.groupby('symbol')['funding_rate_annualised'].agg(['mean','std']).reset_index().sort_values('mean', ascending=True)
ac = [COLORS.get(s.lower(),'#8b949e') for s in sf['symbol']]
ax.barh(sf['symbol'], sf['mean'], xerr=sf['std'], color=ac, alpha=0.8, capsize=4, error_kw={'ecolor':'#8b949e'})
ax.axvline(0, color='#8b949e', linewidth=0.8)
ax.set_title('Mean Annualised Funding Rate by Asset (%)', fontsize=11)
ax.set_xlabel('Annualised Funding (%)'); ax.grid(True, alpha=0.3, axis='x')

plt.suptitle('Funding Rate Analysis', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('funding_analysis.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print(f"Funding→next-day Spearman IC = {ic:.4f}  (p={pval:.4f})")
print("✅ Negative IC: extreme positive funding predicts lower next-day returns" if ic<0 else "Positive IC found")


---
## 3. 💥 Liquidation Cascades — LUNA and FTX Anatomy

| Event | Date | BTC Move | Cascade Multiplier |
|-------|------|----------|--------------------|
| LUNA/UST collapse | May 2022 | −35% | 5× |
| FTX collapse | Nov 2022 | −28% | **18×** |
| USDC depeg | Mar 2023 | −8% | 2× |


In [ ]:
fig, axes = plt.subplots(2,2, figsize=(16,10))

# Panel 1: Daily volume
ax = axes[0,0]
dl = liqs.groupby(['date','side'])['liq_usd'].sum().reset_index()
ll = dl[dl['side']=='long'].set_index('date')['liq_usd']/1e6
sl = dl[dl['side']=='short'].set_index('date')['liq_usd']/1e6
ax.fill_between(ll.index, ll, alpha=0.7, color='#f85149', label='Long liqs (price fell)')
ax.fill_between(sl.index, sl, alpha=0.7, color='#3fb950', label='Short liqs (price rose)')
for _, s in liqs[liqs['shock_event'].notna()][['date','shock_event']].drop_duplicates('shock_event').iterrows():
    ax.axvline(s['date'], color='#f85149', linewidth=1.5, linestyle='--', alpha=0.8)
    ax.text(s['date'], ll.max()*0.8, s['shock_event'].replace('_','
'), fontsize=6.5, color='#f85149', ha='center')
ax.set_title('Daily Liquidation Volume ($M)', fontsize=11)
ax.set_ylabel('$M'); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# Panel 2: By asset
ax = axes[0,1]
ba = liqs.groupby('symbol')['liq_usd'].sum().sort_values(ascending=True)/1e9
bc = [COLORS.get(s.lower(),'#8b949e') for s in ba.index]
ax.barh(ba.index, ba.values, color=bc, alpha=0.85)
ax.set_title('Total Liquidations by Asset ($B)', fontsize=11)
ax.set_xlabel('$B'); ax.grid(True, alpha=0.3, axis='x')
for i,v in enumerate(ba.values): ax.text(v+0.02, i, f'${v:.1f}B', va='center', fontsize=8)

# Panel 3: Cascade vs normal
ax = axes[1,0]
cd = liqs[liqs['is_cascade']==1].groupby('date')['liq_usd'].sum()/1e6
nd = liqs[liqs['is_cascade']==0].groupby('date')['liq_usd'].sum()/1e6
ax.fill_between(nd.index, nd, alpha=0.6, color='#388bfd', label=f'Normal ({(liqs.is_cascade==0).sum():,})')
ax.fill_between(cd.index, cd, alpha=0.8, color='#f85149', label=f'Cascade ({(liqs.is_cascade==1).sum():,})')
ax.set_title('Cascade vs Normal Volume ($M)', fontsize=11)
ax.set_ylabel('$M'); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# Panel 4: Size vs return scatter
ax = axes[1,1]
samp = liqs[liqs['liq_usd']>100000].copy()
ax.scatter(samp['btc_return']*100, np.log10(samp['liq_usd']),
           c=samp['is_cascade'].map({0:'#388bfd',1:'#f85149'}), alpha=0.4, s=8)
ax.axvline(0,  color='#8b949e', linewidth=0.8, linestyle='--')
ax.axvline(-5, color='#f85149', linewidth=1, linestyle='--', alpha=0.7, label='−5% threshold')
ax.set_xlabel('BTC Daily Return (%)'); ax.set_ylabel('Liq Size (log₁₀ USD)')
ax.set_title('Liquidation Size vs BTC Return', fontsize=11)
ax.legend(handles=[mpatches.Patch(color='#388bfd',label='Normal'),
                   mpatches.Patch(color='#f85149',label='Cascade')], fontsize=9)
ax.grid(True, alpha=0.3)

plt.suptitle('Liquidation Cascade Analysis — 2022–2024', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('liquidation_analysis.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()

cas = liqs[liqs.is_cascade==1]; nor = liqs[liqs.is_cascade==0]
print(f"Total liquidated:  ${liqs['liq_usd'].sum()/1e9:.1f}B")
print(f"Cascade avg size:  ${cas['liq_usd'].mean():,.0f}  |  Normal avg: ${nor['liq_usd'].mean():,.0f}  |  Ratio: {cas['liq_usd'].mean()/nor['liq_usd'].mean():.1f}×")


---
## 4. 🔗 On-Chain Metrics — Reading the Blockchain

| Metric | Top signal | Bottom signal |
|--------|-----------|---------------|
| MVRV | > 3.5 | < 1.0 |
| SOPR | > 1.0 (profit taking) | < 1.0 (capitulation) |
| Exchange holdings | Rising | Falling |


In [ ]:
btc_oc = onchain[onchain['symbol']=='BTC'].copy().sort_values('date')
btc_oc = btc_oc.merge(btc[['date','close','regime']], on='date', how='left')

fig, axes = plt.subplots(2,2, figsize=(16,11))

# Panel 1: MVRV
ax1 = axes[0,0]; ax1t = ax1.twinx()
ax1.fill_between(btc_oc['date'], btc_oc['mvrv'], alpha=0.4, color='#f7931a')
ax1.plot(btc_oc['date'], btc_oc['mvrv'], color='#f7931a', linewidth=1.5, label='MVRV')
ax1.axhline(3.5, color='#f85149', linewidth=1, linestyle='--', alpha=0.8, label='Top zone (3.5)')
ax1.axhline(1.0, color='#3fb950', linewidth=1, linestyle='--', alpha=0.8, label='Bottom zone (1.0)')
ax1t.plot(btc_oc['date'], btc_oc['close'], color='#c9d1d9', linewidth=1, alpha=0.4, label='Price')
ax1.set_title('BTC MVRV Ratio vs Price', fontsize=11)
ax1.set_ylabel('MVRV', color='#f7931a'); ax1t.set_ylabel('BTC Price ($)', color='#8b949e')
ax1.legend(loc='upper left', fontsize=8); ax1.grid(True, alpha=0.3)

# Panel 2: SOPR
ax = axes[0,1]
sp = btc_oc['sopr'].rolling(14).mean()
ax.fill_between(btc_oc['date'], sp, 1, where=sp>=1, alpha=0.6, color='#3fb950', label='Profit (>1)')
ax.fill_between(btc_oc['date'], sp, 1, where=sp<1,  alpha=0.6, color='#f85149', label='Capitulation (<1)')
ax.axhline(1.0, color='#8b949e', linewidth=1.2)
ax.set_title('BTC SOPR — 14-day MA', fontsize=11)
ax.set_ylabel('SOPR'); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# Panel 3: Exchange holdings
ax = axes[1,0]; axt = ax.twinx()
ax.plot(btc_oc['date'], btc_oc['exchange_holdings_pct'], color='#f85149', linewidth=2)
ax.fill_between(btc_oc['date'], btc_oc['exchange_holdings_pct'], alpha=0.3, color='#f85149', label='Exch. holdings (%)')
axt.plot(btc_oc['date'], btc_oc['close'], color='#c9d1d9', linewidth=1, alpha=0.4)
ax.set_title('Exchange Holdings vs BTC Price', fontsize=11)
ax.set_ylabel('Supply on Exchanges (%)', color='#f85149'); axt.set_ylabel('BTC Price ($)', color='#8b949e')
ax.grid(True, alpha=0.3)

# Panel 4: IC heatmap
ax = axes[1,1]
mets = ['mvrv','nvt','sopr','exchange_holdings_pct','active_addresses']
fwds = [1,7,30]
cm_  = np.zeros((len(mets), len(fwds)))
bos = btc_oc.sort_values('date').copy()
for j,d in enumerate(fwds):
    bos[f'f{d}'] = bos['close'].pct_change(d).shift(-d)
    for i,m in enumerate(mets):
        v = bos[[m,f'f{d}']].dropna()
        if len(v)>10: cm_[i,j],_ = stats.spearmanr(v[m], v[f'f{d}'])

sns.heatmap(cm_, xticklabels=[f'{d}d fwd' for d in fwds],
            yticklabels=['MVRV','NVT','SOPR','Exch.%','Active addr.'],
            annot=True, fmt='.3f', cmap='RdYlGn', center=0, vmin=-0.5, vmax=0.5,
            ax=ax, linewidths=0.5, cbar_kws={'label':'Spearman IC'})
ax.set_title('On-Chain Metrics → Forward Return IC', fontsize=11)

plt.suptitle('On-Chain Metrics — BTC 2022–2024', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('onchain_analysis.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()

for r in ['bear','crab','bull']:
    sub = btc_oc[btc_oc['regime']==r]
    print(f"{r.upper():6s}: MVRV={sub['mvrv'].mean():.2f}  SOPR={sub['sopr'].mean():.3f}  Exch={sub['exchange_holdings_pct'].mean():.1f}%")


---
## 5. 🔀 Cross-Exchange Spreads & Arbitrage

Normal spread: **1–3 bps**. During stress events: **20–100+ bps**.  
Spread spikes are both a risk signal and an arbitrage opportunity.


In [ ]:
btc_cex = cex[cex['symbol']=='BTC'].copy().sort_values('date')
fig, axes = plt.subplots(2,2, figsize=(16,9))

# Panel 1: Time series
ax = axes[0,0]
ms = btc_cex.groupby('date')['max_cross_spread_bps'].first()
ax.fill_between(ms.index, ms, alpha=0.5, color='#388bfd')
ax.axhline(ms.mean(),        color='#f7931a', linewidth=1.5, linestyle='--', label=f'Mean {ms.mean():.1f} bps')
ax.axhline(ms.quantile(.95), color='#f85149', linewidth=1,   linestyle=':', label=f'95th pct {ms.quantile(.95):.1f} bps')
ax.set_title('BTC Max Cross-Exchange Spread (bps)', fontsize=11)
ax.set_ylabel('bps'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# Panel 2: By exchange
ax = axes[0,1]
es = btc_cex.groupby('exchange')['spread_to_ref_bps'].agg(['mean','std'])
ec = {'Binance':'#f0b90b','Coinbase':'#0052ff','Kraken':'#5741d9','OKX':'#8b949e'}
ax.bar(es.index, es['mean'], yerr=es['std'],
       color=[ec.get(e,'#8b949e') for e in es.index], alpha=0.8, capsize=5, error_kw={'ecolor':'#8b949e'})
ax.axhline(0, color='#8b949e', linewidth=0.8)
ax.set_title('Mean Spread to Reference by Exchange (bps)', fontsize=11)
ax.set_ylabel('bps'); ax.grid(True, alpha=0.3, axis='y')

# Panel 3: Stress vs normal distribution
ax = axes[1,0]
stress_ = btc_cex[btc_cex['is_stress']==1]['max_cross_spread_bps']
norm_   = btc_cex[btc_cex['is_stress']==0]['max_cross_spread_bps']
bins = np.linspace(0, max(stress_.quantile(.99), norm_.quantile(.99)), 50)
ax.hist(norm_,   bins=bins, alpha=0.6, color='#388bfd', density=True, label=f'Normal n={len(norm_):,}')
ax.hist(stress_, bins=bins, alpha=0.7, color='#f85149', density=True, label=f'Stress n={len(stress_):,}')
ax.set_title('Spread Distribution: Stress vs Normal', fontsize=11)
ax.set_xlabel('Max Spread (bps)'); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# Panel 4: Spread vs return
ax = axes[1,1]
sr = btc_cex.groupby('date').agg({'max_cross_spread_bps':'first','is_stress':'first'}).reset_index()
sr = sr.merge(btc[['date','daily_return']], on='date').dropna()
ax.scatter(sr['daily_return']*100, sr['max_cross_spread_bps'],
           c=sr['is_stress'].map({0:'#388bfd',1:'#f85149'}), alpha=0.5, s=12)
ax.axvline(0, color='#8b949e', linewidth=0.8, linestyle='--')
ax.set_xlabel('BTC Daily Return (%)'); ax.set_ylabel('Max Spread (bps)')
ax.set_title('Spread vs BTC Daily Return', fontsize=11)
ax.legend(handles=[mpatches.Patch(color='#388bfd',label='Normal'),mpatches.Patch(color='#f85149',label='Stress')], fontsize=9)
ax.grid(True, alpha=0.3)

plt.suptitle('Cross-Exchange Arbitrage Spread Analysis', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('exchange_spreads.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print(f"Stress: {stress_.mean():.1f} bps  |  Normal: {norm_.mean():.1f} bps  |  Ratio: {stress_.mean()/norm_.mean():.1f}×")


---
## 6. 🤖 Regime Classification — Price vs On-Chain Features

**Key question:** Do on-chain metrics add predictive power beyond price features?  
Using **TimeSeriesSplit** — no data leakage, train always precedes test.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import classification_report, confusion_matrix
import warnings; warnings.filterwarnings('ignore')

PRICE   = ['ret_7d','ret_30d','ret_90d','vol_7d','vol_30d','high_low_range','daily_return','funding_rate_8h','funding_annualised']
ONCHAIN = ['mvrv','nvt','sopr','exchange_holdings','active_addresses']
ALL     = PRICE + ONCHAIN

df = regime_f.dropna(subset=ALL+['regime_label']).sort_values('date').copy()
Xa = df[ALL].values; Xp = df[PRICE].values; y = df['regime_label'].values

tscv = TimeSeriesSplit(n_splits=5)
rf   = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=5)

def cv_acc(X,y,m):
    s=[]
    for tr,te in tscv.split(X): m.fit(X[tr],y[tr]); s.append(m.score(X[te],y[te]))
    return np.mean(s), np.std(s)

ap, sp_ = cv_acc(Xp, y, rf)
aa, sa  = cv_acc(Xa, y, rf)
print(f"Price + Funding only:       {ap:.3f} ± {sp_:.3f}")
print(f"Price + Funding + On-chain: {aa:.3f} ± {sa:.3f}")
print(f"On-chain improvement:       {(aa-ap)*100:+.1f}pp")

split = int(len(df)*0.8)
rf.fit(Xa[:split], y[:split])
yp = rf.predict(Xa[split:])

fig, axes = plt.subplots(1,2, figsize=(16,6))

ax = axes[0]
imp = pd.Series(rf.feature_importances_, index=ALL).sort_values(ascending=True)
ax.barh(imp.index, imp.values,
        color=['#f85149' if f in ONCHAIN else '#388bfd' for f in imp.index], alpha=0.85)
ax.set_title('Feature Importance — Regime Classification', fontsize=11)
ax.set_xlabel('Importance')
ax.legend(handles=[mpatches.Patch(color='#388bfd',label='Price/Funding'),
                   mpatches.Patch(color='#f85149',label='On-chain')], fontsize=9)
ax.grid(True, alpha=0.3, axis='x')

ax = axes[1]
cm_ = confusion_matrix(y[split:], yp)
sns.heatmap(cm_, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Bear','Crab','Bull'], yticklabels=['Bear','Crab','Bull'],
            ax=ax, linewidths=0.5, cbar_kws={'label':'Count'})
ax.set_title(f'Confusion Matrix  (OOS accuracy: {(yp==y[split:]).mean():.1%})', fontsize=11)
ax.set_ylabel('True Regime'); ax.set_xlabel('Predicted Regime')

plt.suptitle('ML Regime Classification', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('regime_classification.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print("\n" + classification_report(y[split:], yp, target_names=['Bear','Crab','Bull']))


---
## 7. 📋 Key Findings

**Regime analysis:** Bear delivered −60%+ for altcoins. BTC beta drives the market (0.68–0.88 for alts).

**Funding rates:** Negative Spearman IC — extreme positive funding predicts lower next-day returns. Reliable mean-reversion signal.

**Liquidations:** FTX generated 18× normal liquidation volume. Cascade events are 10–15× larger than normal in USD terms.

**On-chain:** MVRV < 1 historically marks cycle lows. SOPR < 1 = capitulation = accumulation zone. On-chain features add **+5–8pp** regime classification accuracy beyond price alone.

**Cross-exchange:** Stress events widen spreads 5× on average. Coinbase consistently trades at a premium to Binance.

---

*Dataset & notebook by **Sergey Nefedov** | [github.com/Sergpreneur](https://github.com/Sergpreneur)*  
*If this notebook helped your research, an upvote is appreciated! 🙏*
